In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set()

In [ ]:
from sklearn.model_selection import train_test_split


def prepare_data(seed: int) -> dict[str, pd.DataFrame]:
    np.random.seed(seed)

    x = np.linspace(0, np.pi/2, 300)
    y = np.cos(1.6 * np.pi * x)

    x_samples = np.random.uniform(0, np.pi/2, size=100)
    y_samples = np.cos(1.6 * np.pi * x_samples) + np.random.normal(scale=0.15, size=x_samples.shape)

    dataset = pd.DataFrame({"x": x_samples, "y": y_samples})
    train_val_df, test_df = train_test_split(dataset, test_size=0.25, random_state=seed)
    train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=seed)
    return {
        "train": train_df,
        "val": val_df,
        "test": test_df,
        "true": pd.DataFrame({"x": x, "y": y})
    }

In [ ]:
data = prepare_data(seed=14300631)

In [ ]:
train_df = data["train"]
train_df.info()

In [ ]:
train_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(train_df, x="x", y="y", ax=ax)
sns.lineplot(train_df, x="x", y="y", ax=ax)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()
model.fit(train_df[["x"]], train_df["y"])
train_predictions = model.predict(train_df[["x"]])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
sns.lineplot(train_df, x="x", y="y", ax=ax)

sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
sns.lineplot(x=train_df["x"], y=train_predictions, ax=ax)
ax.legend()
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error


train_mse = mean_squared_error(train_df["y"], train_predictions)
print(f"Train MSE (one feature) = {train_mse}")

In [ ]:
train_df["x2"] = train_df["x"] ** 2

In [ ]:
model = LinearRegression()
model.fit(train_df[["x", "x2"]], train_df["y"])
train_predictions = model.predict(train_df[["x", "x2"]])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
sns.lineplot(train_df, x="x", y="y", ax=ax)

sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
sns.lineplot(x=train_df["x"], y=train_predictions, ax=ax)
ax.legend()
plt.show()

In [ ]:
train_mse = mean_squared_error(train_df["y"], train_predictions)
print(f"Train MSE (two features) = {train_mse}")

In [ ]:
train_df["x3"] = train_df["x"] ** 3

model = LinearRegression()
model.fit(train_df[["x", "x2", "x3"]], train_df["y"])
train_predictions = model.predict(train_df[["x", "x2", "x3"]])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
sns.lineplot(train_df, x="x", y="y", ax=ax)

sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
sns.lineplot(x=train_df["x"], y=train_predictions, ax=ax)
ax.legend()
plt.show()

In [ ]:
train_mse = mean_squared_error(train_df["y"], train_predictions)
print(f"Train MSE (three features) = {train_mse}")

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

MAX_DEGREE = 20


train_mses = {}
for i in range(1, MAX_DEGREE + 1):
    x_poly = PolynomialFeatures(degree=i).fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mses[i] = mean_squared_error(train_df["y"], train_predictions)

In [ ]:
train_mses

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(x=train_mses.keys(), y=train_mses.values(), ax=ax)
sns.lineplot(x=train_mses.keys(), y=train_mses.values(), ax=ax)
ax.set_yscale("log")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [1, 3, 4]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)
    ax.set_title(f"{degree = }, {train_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [5, 6, 7]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)
    ax.set_title(f"{degree = }, {train_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [9, 13, 15]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)
    ax.set_title(f"{degree = }, {train_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [18, 19, 20]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)
    ax.set_title(f"{degree = }, {train_mse = :.5f}")
    ax.legend()
plt.show()

## Добавим валидацию:

In [ ]:
val_df = data["val"]
val_df.info()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [1, 3, 4]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    x_val = poly.fit_transform(val_df[["x"]])
    val_predictions = model.predict(x_val)
    val_mse = mean_squared_error(val_df["y"], val_predictions)

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)

    sns.scatterplot(val_df, x="x", y="y", marker="x", s=100, ax=ax, label="val_samples")
    ax.set_title(f"{degree = }, {train_mse = :.5f}, {val_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [5, 6, 7]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    x_val = poly.fit_transform(val_df[["x"]])
    val_predictions = model.predict(x_val)
    val_mse = mean_squared_error(val_df["y"], val_predictions)

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)

    sns.scatterplot(val_df, x="x", y="y", marker="x", s=100, ax=ax, label="val_samples")
    ax.set_title(f"{degree = }, {train_mse = :.5f}, {val_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [9, 11, 13]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    x_val = poly.fit_transform(val_df[["x"]])
    val_predictions = model.predict(x_val)
    val_mse = mean_squared_error(val_df["y"], val_predictions)

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)

    sns.scatterplot(val_df, x="x", y="y", marker="x", s=100, ax=ax, label="val_samples")
    ax.set_title(f"{degree = }, {train_mse = :.5f}, {val_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, degree in zip(axes, [18, 19, 20]):
    poly = PolynomialFeatures(degree=degree)

    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mse = mean_squared_error(train_df["y"], train_predictions)

    x_space = np.linspace(0, np.pi/2, 100)
    y_space = model.predict(poly.fit_transform(x_space.reshape(-1, 1)))

    x_val = poly.fit_transform(val_df[["x"]])
    val_predictions = model.predict(x_val)
    val_mse = mean_squared_error(val_df["y"], val_predictions)

    sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
    sns.lineplot(train_df, x="x", y="y", ax=ax)

    sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions")
    sns.lineplot(x=x_space, y=y_space, ax=ax)

    sns.scatterplot(val_df, x="x", y="y", marker="x", s=100, ax=ax, label="val_samples")
    ax.set_title(f"{degree = }, {train_mse = :.5f}, {val_mse = :.5f}")
    ax.legend()
plt.show()

In [ ]:
MAX_DEGREE = 20


train_mses = {}
val_mses = {}
for i in range(1, MAX_DEGREE + 1):
    poly = PolynomialFeatures(degree=i)
    x_poly = poly.fit_transform(train_df[["x"]])
    model = LinearRegression().fit(x_poly, train_df["y"])
    train_predictions = model.predict(x_poly)
    train_mses[i] = mean_squared_error(train_df["y"], train_predictions)

    x_val_poly = poly.fit_transform(val_df[["x"]])
    val_predictions = model.predict(x_val_poly)
    val_mses[i] = mean_squared_error(val_df["y"], val_predictions)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(x=train_mses.keys(), y=train_mses.values(), ax=ax, label="train_mse")
sns.lineplot(x=train_mses.keys(), y=train_mses.values(), ax=ax)

sns.scatterplot(x=val_mses.keys(), y=val_mses.values(), ax=ax, label="val_mse")
sns.lineplot(x=val_mses.keys(), y=val_mses.values(), ax=ax)
ax.set_yscale("log")
plt.show()

In [ ]:
best_degree = 5
test_df = data["test"]

poly = PolynomialFeatures(degree=best_degree)
x_train = poly.fit_transform(train_df[["x"]])
model = LinearRegression().fit(x_train, train_df["y"])

train_predictions = model.predict(x_train)
val_predictions = model.predict(poly.fit_transform(val_df[["x"]]))
test_predictions = model.predict(poly.fit_transform(test_df[["x"]]))

train_mse = mean_squared_error(train_df["y"], train_predictions)
val_mse = mean_squared_error(val_df["y"], val_predictions)
test_mse = mean_squared_error(test_df["y"], test_predictions)

fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(train_df, x="x", y="y", ax=ax, label="samples")
sns.scatterplot(val_df, x="x", y="y", marker="x", s=100, ax=ax, label="val_samples", color="green")
sns.scatterplot(test_df, x="x", y="y", marker="x", s=100, ax=ax, label="test_samples", color="red")

sns.scatterplot(x=train_df["x"], y=train_predictions, ax=ax, label="predictions", color="brown")
sns.scatterplot(x=val_df["x"], y=val_predictions, ax=ax, color="brown")
sns.scatterplot(x=test_df["x"], y=test_predictions, ax=ax, color="brown")
ax.set_title(f"{train_mse = :.5f}, {val_mse = :.5f}, {test_mse = :.5f}")
ax.legend()
plt.show()

# Boston Housing dataset

http://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html

In [ ]:
!wget https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv

In [ ]:
df = pd.read_csv("BostonHousing.csv")
df.head()

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split


train_val_df, test_df = train_test_split(df, test_size=0.2)
train_df, val_df = train_test_split(train_val_df, test_size=0.2)

In [ ]:
y_train = train_df["medv"]
X_train = train_df.drop("medv", axis=1)

y_val = val_df["medv"]
X_val = val_df.drop("medv", axis=1)

y_test = test_df["medv"]
X_test = test_df.drop("medv", axis=1)

In [ ]:
model = LinearRegression().fit(X_train, y_train)

train_pred = model.predict(X_train)
val_pred  = model.predict(X_val)

print(f"Train MSE: {mean_squared_error(y_train, train_pred)}")
print(f"Validation MSE: {mean_squared_error(y_val, val_pred)}")

In [ ]:
# визуализация коэффициентов линейной регрессии
def visualize_coefficients(coefs, feature_names: list[str], top_n: int) -> None:
    """Функция для визуализации коэффициентов линейной регрессии.

    Параметры:
        coefs: коэффициенты модели (model.coef_).
        feature_names: названия признаков (X_train.columns).
        top_n: вывести top_n самых положительных и top_n самых отрицательных признаков.
    """
    feature_names = np.array(feature_names)
    if top_n * 2 > len(coefs):
        n_pos = len(coefs) // 2
        n_neg = len(coefs) - n_pos
    else:
        n_pos, n_neg = top_n, top_n
    # нам нужно найти индексы top_n наибольших и top_n наименьших коэффициентов
    min_coef_idxs = np.argsort(coefs)[:n_neg]
    max_coef_idxs = np.argsort(coefs)[len(coefs) - n_pos:]
    # соответствующие имена фичей
    top_feature_names = np.concatenate((feature_names[min_coef_idxs], feature_names[max_coef_idxs]))
    # отобразим на bar-графике
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(np.arange(n_neg), coefs[min_coef_idxs], color=sns.xkcd_rgb["mauve"], hatch="/")
    ax.bar(np.arange(n_neg, n_neg + n_pos), coefs[max_coef_idxs], color=sns.xkcd_rgb["teal"], hatch="\\")
    ax.set_xticks(np.arange(0, n_neg + n_pos))
    ax.set_xticklabels(top_feature_names, rotation=45, ha="right", fontsize=14)
    plt.show()

In [ ]:
visualize_coefficients(model.coef_, X_train.columns, top_n=10)

In [ ]:
from sklearn.preprocessing import StandardScaler


scl = StandardScaler()
X_train_scaled = scl.fit_transform(X_train)
X_val_scaled = scl.transform(X_val)

In [ ]:
model = LinearRegression().fit(X_train_scaled, y_train)

train_pred = model.predict(X_train_scaled)
val_pred  = model.predict(X_val_scaled)

print(f"Train MSE: {mean_squared_error(y_train, train_pred)}")
print(f"Validation MSE: {mean_squared_error(y_val, val_pred)}")

In [ ]:
from sklearn.metrics import mean_squared_error

train_pred = model.predict(X_train_scaled)
val_pred  = model.predict(X_val_scaled)

print(f"Train MSE: {mean_squared_error(y_train, train_pred)}")
print(f"Validation MSE: {mean_squared_error(y_val, val_pred)}")

In [ ]:
visualize_coefficients(model.coef_, X_train.columns, top_n=10)